#BRONZE (v1) – Raw Data

DOWNLOAD DATA

In [75]:
!pip install gdown
import gdown

# File ID
teams_id = "1CKkXndq0dhs3jF2ST9DggH3cruuVC45R"
players_id = "1MFYOr41I3dDU6T6CmRS6zkNhIS8bHnAw"

# Output file
teams_file = "male_teams.csv"
players_file = "male_players.csv"

# Download (quan trọng: fuzzy=True)
gdown.download(id=teams_id, output=teams_file, fuzzy=True)
gdown.download(id=players_id, output=players_file, fuzzy=True)

Downloading...
From (original): https://drive.google.com/uc?id=1CKkXndq0dhs3jF2ST9DggH3cruuVC45R
From (redirected): https://drive.google.com/uc?id=1CKkXndq0dhs3jF2ST9DggH3cruuVC45R&confirm=t&uuid=fa60ed91-9b27-4963-a6c5-3d5a22ff3a9a
To: /content/male_teams.csv
100%|██████████| 113M/113M [00:02<00:00, 54.5MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1MFYOr41I3dDU6T6CmRS6zkNhIS8bHnAw
From (redirected): https://drive.google.com/uc?id=1MFYOr41I3dDU6T6CmRS6zkNhIS8bHnAw&confirm=t&uuid=4c4786ed-5730-4678-a618-8880ce5b8aad
To: /content/male_players.csv
100%|██████████| 5.64G/5.64G [01:18<00:00, 72.1MB/s]


'male_players.csv'

KHỞI TẠO SPARK

In [76]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("FIFA23").getOrCreate()


LOAD DATA (RAW)

In [77]:
teams_df = spark.read.csv(teams_file, header=True, inferSchema=True)
players_df = spark.read.csv(players_file, header=True, inferSchema=True)

KIỂM TRA DỮ LIỆU BAN ĐẦU

In [78]:
teams_df.show(5)
players_df.show(5)

+-------+--------------------+------------+-----------+----------------+-------------------+---------+--------------------+------------+--------------+----------------+-------+------+--------+-------+--------+--------------------+----------+----------------------+-----------------+-------------------+--------------+-----------------------+----------------------+-------+---------------+--------------+--------------------+---------------------+---------+-----------+------------+--------------------+--------------+--------------+--------------------+----------------------+-----------------+-------------------------+---------+-----------------+-------------------+--------------+------------------+-----------+--------------+-------------------+-----------------------+---------------------+-------------------------+-----------------------+------------------------+------------------------+---------------------------+
|team_id|            team_url|fifa_version|fifa_update|fifa_update_date|    

In [79]:
# Tạo version v1 từ raw data
teams_v1 = teams_df
players_v1 = players_df

#SILVER (v2) – Data Cleaning

In [80]:
# VERSION v2 - SILVER (DATA CLEANING)
from pyspark.sql.functions import col, isnan, when, count

# Clone từ v1
teams_v2 = teams_v1
players_v2 = players_v1

## Kiểm tra dữ liệu thiếu (NULL, NaN)

In [81]:
from pyspark.sql.types import NumericType
from pyspark.sql.functions import col, when, count, isnan

# Hàm kiểm tra NULL an toàn
def null_check(df):
    return df.select([
        count(
            when(
                # Nếu là numeric → check NULL + NaN
                (col(c).isNull() | isnan(c)) if isinstance(df.schema[c].dataType, NumericType)
                # Nếu không → chỉ check NULL
                else col(c).isNull(),
                c
            )
        ).alias(c)
        for c in df.columns
    ])

# ÁP DỤNG
print("=== NULL CHECK (PLAYERS) ===")
null_check(players_v2).show()

print("=== NULL CHECK (TEAMS) ===")
null_check(teams_v2).show()

=== NULL CHECK (PLAYERS) ===
+---------+----------+------------+-----------+----------------+----------+---------+----------------+-------+---------+---------+--------+---+---+---------+---------+---------+-----------+------------+------------+---------+-------------+------------------+----------------+----------------+------------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+---------+---------+---------+------------------+-----------+-------------+-------+--------+-------+---------+---------+-------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+-------------+--------------+----------------+--------

##Xử lý null

In [82]:
from pyspark.sql.functions import col
from pyspark.sql.types import NumericType

# players
df_players = players_v2

# 1. Drop dòng thiếu quan trọng
important_cols = ["overall", "potential", "age"]
df_players = df_players.dropna(subset=important_cols)

# 2. Fill numeric = 0
numeric_cols = [
    c for c in df_players.columns
    if isinstance(df_players.schema[c].dataType, NumericType)
]
df_players = df_players.fillna(0, subset=numeric_cols)

# 3. Fill string
string_cols = [
    c for c in df_players.columns
    if df_players.schema[c].dataType.simpleString() == "string"
]
df_players = df_players.fillna("Unknown", subset=string_cols)

players_clean = df_players


# teams
df_teams = teams_v2

# 1. Drop nhẹ (chỉ cột cực quan trọng)
df_teams = df_teams.dropna(subset=["team_name", "overall"])

# 2. Fill numeric = 0
numeric_cols_teams = [
    c for c in df_teams.columns
    if isinstance(df_teams.schema[c].dataType, NumericType)
]
df_teams = df_teams.fillna(0, subset=numeric_cols_teams)

# 3. Fill string
string_cols_teams = [
    c for c in df_teams.columns
    if df_teams.schema[c].dataType.simpleString() == "string"
]
df_teams = df_teams.fillna("Unknown", subset=string_cols_teams)

teams_clean = df_teams

drop duplicate + fill null

In [83]:
from pyspark.sql.functions import col, when, isnan
from pyspark.sql.types import NumericType

# players
# Drop duplicates
players_v2 = players_v2.dropDuplicates(["player_id"])

# Fill null numeric bằng 0
for c in players_v2.columns:
    if isinstance(players_v2.schema[c].dataType, NumericType):
        players_v2 = players_v2.fillna({c: 0})
# Kiểm tra số dòng sau khi xử lý
print("Players after cleaning:", players_v2.count())

# Hiển thị dữ liệu sau khi clean
players_v2.show(5, False)


# teams
teams_v2 = teams_v2.dropDuplicates(["team_id"])

for c in teams_v2.columns:
    if isinstance(teams_v2.schema[c].dataType, NumericType):
        teams_v2 = teams_v2.fillna({c: 0})
print("Teams after cleaning:", teams_v2.count())

teams_v2.show(5, False)

Players after cleaning: 56880
+---------+--------------------------------------+------------+-----------+----------------+------------+--------------------+----------------+-------+---------+---------+--------+---+----------+---------+---------+---------+------------+------------+------------+-------------------+-------------+------------------+----------------+----------------+------------------------------+--------------+----------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-------------+----------------+---------+------------------+---------------------------------+------------------------------------------------------------------------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+

##Xoá dữ liệu trùng (Duplicate)

In [84]:
# players
# Đếm trước khi xoá
print("Players trước khi xoá duplicate:", players_v2.count())

# Xoá duplicate (toàn bộ dòng giống nhau)
players_clean = players_v2.dropDuplicates()

# Đếm sau khi xoá
print("Players sau khi xoá duplicate:", players_clean.count())

# Hiển thị thử
print("=== PLAYERS AFTER DROP DUPLICATE ===")
players_clean.show(5)

# teams
print("Teams trước khi xoá duplicate:", teams_v2.count())

teams_clean = teams_v2.dropDuplicates()

print("Teams sau khi xoá duplicate:", teams_clean.count())

print("=== TEAMS AFTER DROP DUPLICATE ===")
teams_clean.show(5)

Players trước khi xoá duplicate: 56880
Players sau khi xoá duplicate: 56880
=== PLAYERS AFTER DROP DUPLICATE ===
+---------+--------------------+------------+-----------+----------------+---------------+--------------------+----------------+-------+---------+---------+--------+---+----------+---------+---------+---------+--------------+------------+------------+-----------+-------------+------------------+----------------+----------------+------------------------------+--------------+-------------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-------------+----------------+---------+------------------+-----------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------

##Data Validation (Loại bỏ dữ liệu sai)

###Loại bỏ giá trị âm bất hợp lý

In [85]:
from pyspark.sql.functions import col

# players
df_players = players_clean

non_negative_cols = [
    "value_eur", "wage_eur",
    "overall", "potential",
    "age", "height_cm", "weight_kg"
]

for c in non_negative_cols:
    df_players = df_players.filter(col(c) >= 0)

players_clean = df_players

print("Players sau khi loại giá trị âm:", players_clean.count())
players_clean.select("short_name", "value_eur", "wage_eur").show(5)


# teams
df_teams = teams_clean

non_negative_cols_teams = [
    "overall", "attack", "midfield", "defence",
    "transfer_budget_eur", "club_worth_eur"
]

for c in non_negative_cols_teams:
    df_teams = df_teams.filter(col(c) >= 0)

teams_clean = df_teams

print("Teams sau khi loại giá trị âm:", teams_clean.count())
teams_clean.select("team_name", "overall", "club_worth_eur").show(5)

Players sau khi loại giá trị âm: 56880
+---------------+---------+--------+
|     short_name|value_eur|wage_eur|
+---------------+---------+--------+
|     R. Ziegler|  1400000|   25000|
|    L. Rosenior|   775000|    8000|
|      G. Dicker|   425000|    2000|
|     S. Aydoğdu|  1500000|   25000|
|J. Moore-Taylor|    70000|    2000|
+---------------+---------+--------+
only showing top 5 rows
Teams sau khi loại giá trị âm: 1102
+----------------+-------+--------------+
|       team_name|overall|club_worth_eur|
+----------------+-------+--------------+
|         Palermo|     68|      20000000|
|Cúcuta Deportivo|     64|       8300000|
|V-Varen Nagasaki|     62|      10700000|
|             Goa|     60|       2500000|
|         Udinese|     75|     160000000|
+----------------+-------+--------------+
only showing top 5 rows


###Loại bỏ dữ liệu lỗi format

In [86]:
# players
from pyspark.sql.functions import isnan

players_clean.select([
    col(c)
    for c in ["value_eur", "wage_eur", "age"]
]).show(5)

from pyspark.sql.functions import col

df_players = players_clean

numeric_cols = ["value_eur", "wage_eur", "age", "height_cm", "weight_kg"]

for c in numeric_cols:
    df_players = df_players.withColumn(c, col(c).cast("double"))

players_clean = df_players


# teams
df_teams = teams_clean

numeric_cols_teams = [
    "overall", "attack", "midfield", "defence",
    "transfer_budget_eur", "club_worth_eur"
]

for c in numeric_cols_teams:
    df_teams = df_teams.withColumn(c, col(c).cast("double"))

teams_clean = df_teams

+---------+--------+---+
|value_eur|wage_eur|age|
+---------+--------+---+
|  1400000|   25000| 28|
|   775000|    8000| 29|
|   425000|    2000| 33|
|  1500000|   25000| 23|
|    70000|    2000| 20|
+---------+--------+---+
only showing top 5 rows


In [87]:
# Kiểm tra sau khi làm sạch
print("=== PLAYERS SAMPLE ===")
players_clean.show(5)

print("=== TEAMS SAMPLE ===")
teams_clean.show(5)

=== PLAYERS SAMPLE ===
+---------+--------------------+------------+-----------+----------------+---------------+--------------------+----------------+-------+---------+---------+--------+----+----------+---------+---------+---------+--------------+------------+------------+-----------+-------------+------------------+----------------+----------------+------------------------------+--------------+-------------------+--------------+---------------+--------------------+--------------+---------+-----------+------------------------+-------------+----------------+---------+------------------+-----------+--------------------+----+--------+-------+---------+---------+------+------------------+-------------------+--------------------------+-----------------------+-----------------+---------------+-----------+-----------------+------------------+------------------+---------------------+---------------------+----------------+------------------+----------------+----------------+-------------+----

##Data Standardization (Chuẩn hoá kiểu dữ liệu)

###Chuẩn hoá date -> datetime

In [88]:
# players
from pyspark.sql.functions import to_date, col

df_players = players_clean

df_players = df_players.withColumn(
    "fifa_update_date",
    to_date(col("fifa_update_date"))
).withColumn(
    "dob",
    to_date(col("dob"))
)

players_clean = df_players


# teams
df_teams = teams_clean

df_teams = df_teams.withColumn(
    "fifa_update_date",
    to_date(col("fifa_update_date"))
)

teams_clean = df_teams

###Chuẩn hoá String -> Number

In [89]:
# players
from pyspark.sql.functions import col

numeric_cols_players = [
    "value_eur", "wage_eur",
    "height_cm", "weight_kg",
    "overall", "potential",
    "age"
]

for c in numeric_cols_players:
    df_players = df_players.withColumn(c, col(c).cast("double"))

players_clean = df_players


# teams
numeric_cols_teams = [
    "overall", "attack", "midfield", "defence",
    "transfer_budget_eur", "club_worth_eur",
    "starting_xi_average_age", "whole_team_average_age"
]

for c in numeric_cols_teams:
    df_teams = df_teams.withColumn(c, col(c).cast("double"))

teams_clean = df_teams

###Kết quả sau khi chuẩn hoá

In [90]:
print("=== SCHEMA PLAYERS ===")
players_clean.printSchema()

print("=== SCHEMA TEAMS ===")
teams_clean.printSchema()

=== SCHEMA PLAYERS ===
root
 |-- player_id: integer (nullable = false)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = false)
 |-- fifa_update: integer (nullable = false)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: double (nullable = false)
 |-- potential: double (nullable = false)
 |-- value_eur: double (nullable = false)
 |-- wage_eur: double (nullable = false)
 |-- age: double (nullable = false)
 |-- dob: date (nullable = true)
 |-- height_cm: double (nullable = false)
 |-- weight_kg: double (nullable = false)
 |-- league_id: integer (nullable = false)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = false)
 |-- club_team_id: integer (nullable = false)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nul

##Tiêu chuẩn hoá

In [91]:
print(players_v2.printSchema())

root
 |-- player_id: integer (nullable = false)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = false)
 |-- fifa_update: integer (nullable = false)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = false)
 |-- potential: integer (nullable = false)
 |-- value_eur: integer (nullable = false)
 |-- wage_eur: integer (nullable = false)
 |-- age: integer (nullable = false)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = false)
 |-- weight_kg: integer (nullable = false)
 |-- league_id: integer (nullable = false)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = false)
 |-- club_team_id: integer (nullable = false)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nullable = false)
 

###Feature selection

In [92]:
# players
player_features = [
    "overall", "potential", "value_eur", "wage_eur",
    "age", "height_cm", "weight_kg",
    "pace", "shooting", "passing",
    "dribbling", "defending", "physic"
]
players_v2.select(player_features).show(5)


# teams
team_features = [
    "overall", "attack", "midfield", "defence",
    "transfer_budget_eur", "club_worth_eur",
    "starting_xi_average_age", "whole_team_average_age"
]
teams_v2.select(team_features).show(5)

+-------+---------+---------+--------+---+---------+---------+----+--------+-------+---------+---------+------+
|overall|potential|value_eur|wage_eur|age|height_cm|weight_kg|pace|shooting|passing|dribbling|defending|physic|
+-------+---------+---------+--------+---+---------+---------+----+--------+-------+---------+---------+------+
|     81|       81| 27500000|   21000| 27|      173|       63|  79|      82|     79|       80|       43|    70|
|     79|       79|  5500000|   76000| 34|      188|       77|  49|      32|     61|       58|       82|    75|
|     77|       77| 11000000|   68000| 28|      185|       80|  78|      78|     61|       76|       36|    76|
|     77|       77| 10000000|   56000| 27|      176|       79|  77|      63|     75|       78|       73|    63|
|     76|       76|  3500000|   19000| 34|      175|       70|  76|      75|     72|       78|       42|    62|
+-------+---------+---------+--------+---+---------+---------+----+--------+-------+---------+---------+

#GOLD LAYER (VECTOR ASSEMBLER)

##Vector assembler

In [93]:
# players
from pyspark.ml.feature import VectorAssembler

assembler_p = VectorAssembler(
    inputCols=player_features,
    outputCol="features"
)

gold_players = assembler_p.transform(players_v2).select("player_id", "features")
gold_players.select("player_id", "features").show(5, False)


# teams
assembler_t = VectorAssembler(
    inputCols=team_features,
    outputCol="features"
)

gold_teams = assembler_t.transform(teams_v2).select("team_id", "features")
gold_teams.select("team_id", "features").show(5, False)

+---------+---------------------------------------------------------------------------+
|player_id|features                                                                   |
+---------+---------------------------------------------------------------------------+
|213516   |[81.0,81.0,2.75E7,21000.0,27.0,173.0,63.0,79.0,82.0,79.0,80.0,43.0,70.0]   |
|169588   |[79.0,79.0,5500000.0,76000.0,34.0,188.0,77.0,49.0,32.0,61.0,58.0,82.0,75.0]|
|204529   |[77.0,77.0,1.1E7,68000.0,28.0,185.0,80.0,78.0,78.0,61.0,76.0,36.0,76.0]    |
|210736   |[77.0,77.0,1.0E7,56000.0,27.0,176.0,79.0,77.0,63.0,75.0,78.0,73.0,63.0]    |
|182945   |[76.0,76.0,3500000.0,19000.0,34.0,175.0,70.0,76.0,75.0,72.0,78.0,42.0,62.0]|
+---------+---------------------------------------------------------------------------+
only showing top 5 rows
+-------+-----------------------------------------------------+
|team_id|features                                             |
+-------+-----------------------------------------------

##Scaling

In [94]:
# players
from pyspark.ml.feature import StandardScaler

scaler_p = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model_p = scaler_p.fit(gold_players)
players_scaled = scaler_model_p.transform(gold_players)
players_scaled.select("player_id", "scaled_features").show(5, False)


# teams
scaler_t = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withMean=True,
    withStd=True
)

scaler_model_t = scaler_t.fit(gold_teams)
teams_scaled = scaler_model_t.transform(gold_teams)
teams_scaled.select("team_id", "scaled_features").show(5, False)

+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|player_id|scaled_features                                                                                                                                                                                                                                                |
+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|213516   |[2.6925400254412373,2.2223501151390805,5.765543624758197,1.2443497025557912,0.3671985682328044,-1.168400588185263,-1.7004684976354572,0.8758682279189693,1.8455641908564646,1.56006489534

#MODELS (SVD = PCA trong Spark)

In [95]:
# players
from pyspark.ml.feature import PCA

pca_players = PCA(
    k=3,
    inputCol="scaled_features",
    outputCol="svd_features"
)

players_model = pca_players.fit(players_scaled)
players_svd = players_model.transform(players_scaled)
players_svd.select("player_id", "svd_features").show(5, False)


# teams
pca_teams = PCA(
    k=2,
    inputCol="scaled_features",
    outputCol="svd_features"
)

teams_model = pca_teams.fit(teams_scaled)
teams_svd = teams_model.transform(teams_scaled)
teams_svd.select("team_id", "svd_features").show(5, False)

+---------+-------------------------------------------------------------+
|player_id|svd_features                                                 |
+---------+-------------------------------------------------------------+
|213516   |[-4.744067901041401,-3.5653699197971327,-3.55607849637208]   |
|169588   |[-2.3701234108108498,-5.449887147782438,-0.2991133019084158] |
|204529   |[-3.317217137129684,-4.708335745099345,-1.3887634828109678]  |
|210736   |[-3.758959332034673,-3.7228549586762814,-1.3728179350834615] |
|182945   |[-3.1064644132148294,-1.5050824792955009,-0.4603622093675711]|
+---------+-------------------------------------------------------------+
only showing top 5 rows
+-------+----------------------------------------+
|team_id|svd_features                            |
+-------+----------------------------------------+
|463    |[-1.1695532950644059,0.7085171222232041]|
|110081 |[-1.000291872155305,-1.9070165489550877]|
|1829   |[1.7917200755249847,-0.5558331607121179]|
|195